# Mutual Fund Analytics Platform — Data Exploration
**Week 1 review notebook.** Connects to local PostgreSQL and visualises key datasets.
Run after the full pipeline: ingestion → cleaning → ETL → metrics → SQL layer.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# Load credentials from .env
load_dotenv(Path('..') / '.env')

conn = psycopg2.connect(
    host=os.getenv('LOCAL_DB_HOST', 'localhost'),
    port=int(os.getenv('LOCAL_DB_PORT', '5432')),
    dbname=os.getenv('LOCAL_DB_NAME', 'mf_analytics'),
    user=os.getenv('LOCAL_DB_USER', 'postgres'),
    password=os.getenv('LOCAL_DB_PASSWORD', ''),
)

def sql(query: str) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(query)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

print('Connected to', os.getenv('LOCAL_DB_NAME'))

## 1. Fund Metrics Overview

In [ ]:
funds = sql("""
    SELECT scheme_code, fund_name, source, is_benchmark,
           ROUND(cagr_1y::numeric, 2)     AS cagr_1y,
           ROUND(cagr_3y::numeric, 2)     AS cagr_3y,
           ROUND(cagr_5y::numeric, 2)     AS cagr_5y,
           ROUND(std_dev_1y::numeric, 2)  AS vol_1y,
           ROUND(max_drawdown::numeric,2) AS max_dd,
           ROUND(sharpe_ratio::numeric,3) AS sharpe,
           ROUND(beta::numeric, 4)        AS beta,
           ROUND(alpha::numeric, 4)       AS alpha
    FROM dbo.vw_fund_performance
    ORDER BY cagr_1y DESC NULLS LAST
""")
print(f'{len(funds)} funds with metrics')
funds[['scheme_code','cagr_1y','cagr_3y','cagr_5y','vol_1y','sharpe','beta','alpha']]

## 2. NAV Time Series — NIFTYBEES.NS vs GOLDBEES.NS

In [ ]:
nav = sql("""
    SELECT df.scheme_code, dd.full_date, fn.nav
    FROM dbo.Fact_NAV fn
    JOIN dbo.Dim_Fund df ON df.fund_key = fn.fund_key
    JOIN dbo.Dim_Date dd ON dd.date_key = fn.date_key
    WHERE df.scheme_code IN ('NIFTYBEES.NS', 'GOLDBEES.NS')
    ORDER BY df.scheme_code, dd.full_date
""")
nav['full_date'] = pd.to_datetime(nav['full_date'])
nav['nav'] = nav['nav'].astype(float)

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()

for code, ax, color in [
    ('NIFTYBEES.NS', ax1, '#1f77b4'),
    ('GOLDBEES.NS',  ax2, '#ff7f0e'),
]:
    d = nav[nav['scheme_code'] == code]
    ax.plot(d['full_date'], d['nav'], color=color, linewidth=1.2, label=code)
    ax.set_ylabel(f'{code} NAV (INR)', color=color)
    ax.tick_params(axis='y', labelcolor=color)

ax1.set_title('NAV Time Series: Nifty 50 ETF vs Gold ETF (2021-2026)', fontweight='bold')
ax1.set_xlabel('Date')
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc='upper left')
plt.tight_layout()
plt.savefig('../docs/screenshots/nav_timeseries.png', bbox_inches='tight')
plt.show()

## 3. Risk-Return Scatter — All 16 Funds

In [ ]:
rr = funds.dropna(subset=['cagr_1y', 'vol_1y']).copy()
rr['cagr_1y'] = rr['cagr_1y'].astype(float)
rr['vol_1y']  = rr['vol_1y'].astype(float)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if bool(r) else '#1f77b4' for r in rr['is_benchmark']]
ax.scatter(rr['vol_1y'], rr['cagr_1y'], c=colors, s=80, alpha=0.8, zorder=3)

for _, row in rr.iterrows():
    ax.annotate(
        row['scheme_code'].replace('.NS','').replace('^',''),
        (float(row['vol_1y']), float(row['cagr_1y'])),
        textcoords='offset points', xytext=(6, 3), fontsize=7
    )

ax.axhline(6.5, color='grey', linestyle='--', linewidth=0.8, label='Rf = 6.5%')
ax.axhline(0,   color='black', linestyle='-',  linewidth=0.4)
ax.set_xlabel('Annualised Volatility 1Y (%)')
ax.set_ylabel('1Y Return (%)')
ax.set_title('Risk-Return Scatter — All 16 Funds (Trailing 1Y)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/screenshots/risk_return_scatter.png', bbox_inches='tight')
plt.show()

## 4. Sharpe Ratio Bar Chart — Ranked

In [ ]:
sharpe = funds.dropna(subset=['sharpe']).copy()
sharpe['sharpe'] = sharpe['sharpe'].astype(float)
# Exclude LIQUIDBEES extreme value for readable chart
sharpe = sharpe[sharpe['scheme_code'] != 'LIQUIDBEES.NS'].sort_values('sharpe', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2ca02c' if float(v) > 0 else '#d62728' for v in sharpe['sharpe']]
bars = ax.barh(sharpe['scheme_code'], sharpe['sharpe'], color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Sharpe Ratio (trailing 1Y, Rf=6.5%)')
ax.set_title('Sharpe Ratio Ranking — 15 Funds (LIQUIDBEES excluded)', fontweight='bold')
for bar, val in zip(bars, sharpe['sharpe']):
    ax.text(float(val) + 0.02, bar.get_y() + bar.get_height()/2,
            f'{float(val):.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../docs/screenshots/sharpe_ranking.png', bbox_inches='tight')
plt.show()

## 5. Investor Segmentation

In [ ]:
inv = sql("""
    SELECT investor_segment, risk_profile, COUNT(*) AS n
    FROM dbo.vw_investor_segmentation
    GROUP BY investor_segment, risk_profile
    ORDER BY investor_segment, risk_profile
""")
inv['n'] = inv['n'].astype(int)

pivot = inv.pivot(index='investor_segment', columns='risk_profile', values='n').fillna(0)
pivot = pivot.reindex(columns=['Conservative','Moderate','Aggressive'])
pivot = pivot.reindex(['Retail','HNI','Institutional'])

fig, ax = plt.subplots(figsize=(8, 4))
pivot.plot(kind='bar', ax=ax, color=['#1f77b4','#ff7f0e','#2ca02c'], alpha=0.85)
ax.set_xlabel('Investor Segment')
ax.set_ylabel('Number of Investors')
ax.set_title('Investor Segmentation by Risk Profile (500 Synthetic Investors)', fontweight='bold')
ax.legend(title='Risk Profile')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../docs/screenshots/investor_segmentation.png', bbox_inches='tight')
plt.show()
print(pivot.to_string())

## 6. Monthly SIP Volume

In [ ]:
sip = sql("""
    SELECT dd.full_date, SUM(fs.monthly_sip_amount) AS total_sip
    FROM dbo.Fact_SIP fs
    JOIN dbo.Dim_Date dd ON dd.date_key = fs.date_key
    GROUP BY dd.full_date
    ORDER BY dd.full_date
""")
sip['full_date'] = pd.to_datetime(sip['full_date'])
sip['total_sip'] = sip['total_sip'].astype(float)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(sip['full_date'], sip['total_sip'] / 1e6, color='#1f77b4', alpha=0.7, width=20)
ax.set_xlabel('Month')
ax.set_ylabel('Total SIP Inflow (INR Millions)')
ax.set_title('Monthly SIP Volume — All 500 Investors (36-Month History)', fontweight='bold')
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1fM'))
plt.tight_layout()
plt.savefig('../docs/screenshots/sip_monthly.png', bbox_inches='tight')
plt.show()
print(f"Total SIP invested: INR {sip['total_sip'].sum():,.0f}")

## 7. Top 10 Funds by AUM

In [ ]:
aum = sql("SELECT * FROM dbo.sp_compute_aum('2026-05-28') LIMIT 10")
aum['aum_inr'] = aum['aum_inr'].astype(float)

fig, ax = plt.subplots(figsize=(10, 5))
labels = aum['fund_name'].str[:45]  # truncate long names
ax.barh(labels[::-1], aum['aum_inr'][::-1] / 1e6, color='#17becf', alpha=0.85)
ax.set_xlabel('AUM (INR Millions)')
ax.set_title('Top 10 Funds by AUM — As of 2026-05-28 (Synthetic Data)', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.FormatStrFormatter('%.1fM'))
plt.tight_layout()
plt.savefig('../docs/screenshots/top10_aum.png', bbox_inches='tight')
plt.show()
print(f"Total AUM tracked: INR {aum['aum_inr'].sum():,.0f}")

## 8. Week 1 Pipeline Summary

In [ ]:
summary = sql("""
    SELECT 'Dim_Date'           AS table_name, COUNT(*) AS rows FROM dbo.Dim_Date
    UNION ALL SELECT 'Dim_AMC',          COUNT(*) FROM dbo.Dim_AMC
    UNION ALL SELECT 'Dim_Category',     COUNT(*) FROM dbo.Dim_Category
    UNION ALL SELECT 'Dim_Fund',         COUNT(*) FROM dbo.Dim_Fund
    UNION ALL SELECT 'Dim_Investor',     COUNT(*) FROM dbo.Dim_Investor
    UNION ALL SELECT 'Fact_NAV',         COUNT(*) FROM dbo.Fact_NAV
    UNION ALL SELECT 'Fact_Transactions',COUNT(*) FROM dbo.Fact_Transactions
    UNION ALL SELECT 'Fact_SIP',         COUNT(*) FROM dbo.Fact_SIP
    UNION ALL SELECT 'Fact_Returns',     COUNT(*) FROM dbo.Fact_Returns
""")
summary['rows'] = summary['rows'].astype(int)
print('Week 1 schema row counts:')
print(summary.to_string(index=False))

conn.close()
print('\nNotebook complete. DB connection closed.')